# Venue → Geohash9 Association

For each TAC venue in `venues-boundaries.geo.json`, find every level-9 geohash cell that is either **fully contained** in the venue's boundary polygon, or **at least 80% covered** by it.

The output is a long-format CSV (one row per venue/geohash pair) saved to `data/venues/tac-list/`.

In [1]:
import json
from collections import deque
from pathlib import Path

import geohash
import geopandas as gpd
import pandas as pd
from shapely.geometry import box, shape
from tqdm.notebook import tqdm

In [2]:
DATA_DIR = Path("../../data/venues/tac-list")
INPUT_PATH = DATA_DIR / "venues-boundaries.geo.json"
OUTPUT_PATH = DATA_DIR / "venues_geohash9.csv"

GEOHASH_PRECISION = 9
COVERAGE_THRESHOLD = 0.8  # a geohash cell is associated with a venue if >= 80% of its area is covered

## Load venue boundaries

One polygon in the source file has an unclosed ring (its first and last coordinates don't match), which GDAL/pyogrio rejects outright. We parse the GeoJSON manually and close any open rings before building the `GeoDataFrame`.

In [3]:
def close_ring(ring):
    """Ensure a linear ring's first and last coordinates match."""
    if ring[0] != ring[-1]:
        return ring + [ring[0]]
    return ring


def fix_geometry(geometry):
    """Close any open rings in a Polygon/MultiPolygon geometry dict."""
    if geometry["type"] == "Polygon":
        rings = [close_ring(ring) for ring in geometry["coordinates"]]
        return {"type": "Polygon", "coordinates": rings}
    if geometry["type"] == "MultiPolygon":
        polygons = [[close_ring(ring) for ring in poly] for poly in geometry["coordinates"]]
        return {"type": "MultiPolygon", "coordinates": polygons}
    return geometry


with open(INPUT_PATH) as f:
    raw_geojson = json.load(f)

properties = [feature["properties"] for feature in raw_geojson["features"]]
geometries = [shape(fix_geometry(feature["geometry"])) for feature in raw_geojson["features"]]

venues_gdf = gpd.GeoDataFrame(properties, geometry=geometries, crs="EPSG:4326")
venues_gdf[["id", "venue_name"]].head()

,id,venue_name
0,1,InterAccess
1,2,401 Richmond
2,3,"918 Bathurst Centre for Culture, Arts, Media a..."
3,4,Albion Library
4,5,Array Space


## Find covering geohash9 cells

For each venue polygon we flood-fill outward from a point known to be inside it, expanding through geohash neighbours as long as they overlap the polygon at all. Any cell whose overlap with the polygon covers at least `COVERAGE_THRESHOLD` of the cell's own area (which includes cells fully contained in the venue) is kept.

This is much faster than scanning every geohash cell in the venue's bounding box, since it only ever visits cells adjacent to ones that already intersect the polygon.

In [4]:
def geohash_cell_polygon(gh):
    """Build the lat/lon rectangle a geohash string represents."""
    cell_bbox = geohash.bbox(gh)
    return box(cell_bbox["w"], cell_bbox["s"], cell_bbox["e"], cell_bbox["n"])


def find_covering_geohashes(geometry, precision=GEOHASH_PRECISION, coverage_threshold=COVERAGE_THRESHOLD):
    """Return {geohash: coverage_fraction} for cells covering >= coverage_threshold of the geometry."""
    parts = geometry.geoms if geometry.geom_type == "MultiPolygon" else [geometry]

    visited = set()
    queue = deque()
    for part in parts:
        seed_point = part.representative_point()
        queue.append(geohash.encode(seed_point.y, seed_point.x, precision=precision))

    results = {}
    while queue:
        gh = queue.popleft()
        if gh in visited:
            continue
        visited.add(gh)

        cell = geohash_cell_polygon(gh)
        if not cell.intersects(geometry):
            continue

        coverage = cell.intersection(geometry).area / cell.area
        if coverage >= coverage_threshold:
            results[gh] = coverage

        for neighbor in geohash.neighbors(gh):
            if neighbor not in visited:
                queue.append(neighbor)

    return results

## Compute geohash9 associations for every venue

In [5]:
rows = []
for venue in tqdm(venues_gdf.itertuples(), total=len(venues_gdf), desc="Venues"):
    covering = find_covering_geohashes(venue.geometry)
    for gh, coverage in covering.items():
        rows.append(
            {
                "venue_id": venue.id,
                "venue_name": venue.venue_name,
                "geohash9": gh,
                "coverage_fraction": round(coverage, 4),
            }
        )

geohash_df = pd.DataFrame(rows)
geohash_df.head()

Venues:   0%|          | 0/85 [00:00<?, ?it/s]

,venue_id,venue_name,geohash9,coverage_fraction
0,1,InterAccess,dpz828v8v,1.0
1,1,InterAccess,dpz828v8u,1.0
2,1,InterAccess,dpz828v8y,1.0
3,1,InterAccess,dpz828v8t,1.0
4,1,InterAccess,dpz828v8s,1.0


## Sanity check

Every venue should have at least one associated geohash9 cell.

In [6]:
geohash_counts = geohash_df.groupby(["venue_id", "venue_name"]).size().rename("num_geohashes")

missing_venues = set(venues_gdf["id"]) - set(geohash_df["venue_id"])
print(f"Venues: {len(venues_gdf)} | Venues with >=1 geohash: {geohash_counts.shape[0]} | Missing: {len(missing_venues)}")
geohash_counts.describe()

Venues: 85 | Venues with >=1 geohash: 85 | Missing: 0


count     85.000000
mean     109.576471
std      149.001468
min        2.000000
25%       23.000000
50%       46.000000
75%      127.000000
max      835.000000
Name: num_geohashes, dtype: float64

## Save output

Long-format CSV: one row per (venue, geohash9) pair, grouped by `venue_id` to recover each venue's full geohash list.

In [7]:
geohash_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(geohash_df)} rows to {OUTPUT_PATH.resolve()}")

Saved 9314 rows to /home/aniket/Programming/eddit-tac/data/venues/tac-list/venues_geohash9.csv
